<a href="https://colab.research.google.com/github/IvanMorsin/Forecasting-electrical-power-in-multi-storey-residential-buildings/blob/main/notebook_10_neural_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install neuralforecast -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.0/287.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 67.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires tornado==6.5.1, but you have tornado 6.5.5 which is incompatible.


In [2]:
!pip install kaleido==0.2.1 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 29.7 MB/s eta 0:00:00


In [3]:
!pip install workalendar -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 57.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 5.2 MB/s eta 0:00:00


In [4]:
!pip install optuna -q

In [5]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from workalendar.europe import Russia
from tqdm import tqdm
import os
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
HOUSE_META = {
    'house_1': {'n_flats': 383, 'n_floors': 12},
    'house_2': {'n_flats': 191, 'n_floors': 12},
    'house_3': {'n_flats': 124, 'n_floors': 12},
    'house_4': {'n_flats': 263, 'n_floors': 12},
    'house_5': {'n_flats': 127, 'n_floors':  7},
    'house_6': {'n_flats': 497, 'n_floors': 25},
    'house_7': {'n_flats': 471, 'n_floors': 17},
    'house_8': {'n_flats': 171, 'n_floors': 23},
}
HOUSES = list(HOUSE_META.keys())

# нормировка n_flats и n_floors для LSTM
n_flats_max = max(m['n_flats'] for m in HOUSE_META.values())
n_floors_max = max(m['n_floors'] for m in HOUSE_META.values())

def save_predictions(ts, house_ids, y_true, y_pred, horizon_name, model_name, filename):
    df_pred = pd.DataFrame({
        'timestamp': ts,
        'house_id': house_ids,
        'y_true': y_true,
        'y_pred': y_pred,
        'horizon': horizon_name,
        'model': model_name,
    })
    if os.path.exists(filename):
        df_existing = pd.read_csv(filename)
        df_pred = pd.concat([df_existing, df_pred], ignore_index=True)
    df_pred.to_csv(filename, index=False)

In [7]:
df = pd.read_csv("df_final+whether.csv", parse_dates=["timestamp"])
houses = [col for col in df.columns if col.startswith("house_")]

cal = Russia()
def is_holiday(dt):
    if dt.weekday() >= 5:
        return 0
    return int(not cal.is_working_day(dt.date()))

df["is_holiday"] = df["timestamp"].apply(is_holiday)

def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    return {"MAE": round(mae, 3), "RMSE": round(rmse, 3), "MAPE": round(mape, 3)}

horizons = {
    "4h": 8,
    "8h": 16,
    "24h": 48,
    "7d": 336,
    "14d": 672,
    "1m": 1488,
}

n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)
input_window = 336

In [8]:
class TimeSeriesDatasetMV(Dataset):
    def __init__(self, data, input_window, horizon):
        self.data = data
        self.input_window = input_window
        self.horizon = horizon

    def __len__(self):
        return len(self.data) - self.input_window - self.horizon + 1

    def __getitem__(self, idx):
        x = self.data[idx: idx + self.input_window]
        y = self.data[idx + self.input_window: idx + self.input_window + self.horizon, 0]
        return torch.FloatTensor(x), torch.FloatTensor(y)

In [9]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=19, hidden_size=128, num_layers=2,
                 horizon=48, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_size, horizon)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        return self.fc(self.dropout(lstm_out[:, -1, :]))

In [10]:
scalers = {}

def prepare_data_all():
    all_frames = []

    df_features = df.copy()
    df_features["hour"] = df_features["timestamp"].dt.hour
    df_features["weekday"] = df_features["timestamp"].dt.weekday
    df_features["month"] = df_features["timestamp"].dt.month
    df_features["day_of_year"] = df_features["timestamp"].dt.dayofyear
    df_features["is_weekend"] = (df_features["weekday"] >= 5).astype(int)

    df_features["month_sin"] = np.sin(2 * np.pi * df_features["month"] / 12)
    df_features["month_cos"] = np.cos(2 * np.pi * df_features["month"] / 12)

    df_features["week_of_year"] = df_features["timestamp"].dt.isocalendar().week
    week_norm = (df_features["week_of_year"] - 1) / 52.0
    df_features["week_sin"] = np.sin(2 * np.pi * week_norm)
    df_features["week_cos"] = np.cos(2 * np.pi * week_norm)

    temp_min = df["temp_c"].iloc[:train_end].min()
    temp_max = df["temp_c"].iloc[:train_end].max()
    temp_scaled = ((df["temp_c"] - temp_min) / (temp_max - temp_min)).values

    for house, meta in HOUSE_META.items():
        power_per_flat = df[[house]].values / meta['n_flats']

        scaler = MinMaxScaler()
        scaler.fit(power_per_flat[:train_end])
        scalers[house] = scaler
        power_scaled = scaler.transform(power_per_flat).flatten()

        n_flats_norm  = meta["n_flats"] / n_flats_max
        n_floors_norm = meta["n_floors"] / n_floors_max

        lag_336 = df[house].shift(336).bfill().values / meta['n_flats']
        lag_672 = df[house].shift(672).bfill().values / meta['n_flats']
        lag_1488 = df[house].shift(1488).bfill().values / meta['n_flats']

        lag_336_scaled = scaler.transform(lag_336.reshape(-1, 1)).flatten()
        lag_672_scaled = scaler.transform(lag_672.reshape(-1, 1)).flatten()
        lag_1488_scaled = scaler.transform(lag_1488.reshape(-1, 1)).flatten()

        rolling_mean_336 = df[house].shift(1).rolling(336).mean().bfill().values / meta['n_flats']
        rolling_mean_672 = df[house].shift(1).rolling(672).mean().bfill().values / meta['n_flats']
        rolling_mean_1488 = df[house].shift(1).rolling(1488).mean().bfill().values / meta['n_flats']

        rolling_mean_336_scaled = scaler.transform(rolling_mean_336.reshape(-1, 1)).flatten()
        rolling_mean_672_scaled = scaler.transform(rolling_mean_672.reshape(-1, 1)).flatten()
        rolling_mean_1488_scaled = scaler.transform(rolling_mean_1488.reshape(-1, 1)).flatten()

        frame = np.column_stack([
            power_scaled,
            df_features["hour"].values / 23.0,
            df_features["weekday"].values / 6.0,
            df_features["month"].values / 12.0,
            temp_scaled,
            df_features["is_weekend"].values.astype(float),
            df["is_holiday"].astype(float).values,
            np.full(len(df), n_flats_norm),
            np.full(len(df), n_floors_norm),
            df_features["month_sin"].values,
            df_features["month_cos"].values,
            df_features["week_sin"].values,
            df_features["week_cos"].values,
            lag_336_scaled,
            lag_672_scaled,
            lag_1488_scaled,
            rolling_mean_336_scaled,
            rolling_mean_672_scaled,
            rolling_mean_1488_scaled,
        ]).astype(np.float32)

        all_frames.append((house, frame))

    return all_frames

INPUT_SIZE = 19
all_house_data = prepare_data_all()

def train_lstm(model, loader_train, loader_val, epochs=100, lr=0.001, patience=10):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    best_val_loss = float("inf")
    patience_counter = 0
    best_weights = None

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for x_batch, y_batch in loader_train:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x_batch, y_batch in loader_val:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                val_loss += criterion(model(x_batch), y_batch).item()

        train_loss /= len(loader_train)
        val_loss /= len(loader_val)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    model.load_state_dict(best_weights)
    return model

def predict_lstm(model, val_data, test_data, input_window, horizon, scaler):
    model.eval()
    predictions, actuals = [], []
    context = np.concatenate([val_data, test_data])

    with torch.no_grad():
        for i in range(0, len(context) - input_window - horizon + 1, horizon):
            x = context[i: i + input_window]
            y = context[i + input_window: i + input_window + horizon, 0]
            if len(y) < horizon:
                break
            x_tensor = torch.FloatTensor(x).unsqueeze(0).to(device)
            y_pred = model(x_tensor).cpu().numpy().flatten()
            predictions.extend(y_pred)
            actuals.extend(y)

    predictions = scaler.inverse_transform(np.array(predictions).reshape(-1, 1)).flatten()
    actuals = scaler.inverse_transform(np.array(actuals).reshape(-1, 1)).flatten()
    return actuals, predictions

In [11]:
# подбор гиперпараметров через Optuna (на горизонте 24h)
def objective(trial, horizon_points):
    hidden_size = trial.suggest_categorical("hidden_size", [64, 128, 256])
    num_layers = trial.suggest_int("num_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr = trial.suggest_float("lr", 1e-3, 1e-2, log=True)  # нижняя граница 0.001
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    loaders_train = []
    loaders_val = []
    for house, frame in all_house_data:
        train_h = frame[:train_end]
        val_h = frame[train_end:val_end]

        if len(train_h) <= input_window + horizon_points:
            continue
        if len(val_h) <= input_window + horizon_points:
            continue

        ds_train = TimeSeriesDatasetMV(train_h, input_window, horizon_points)
        ds_val = TimeSeriesDatasetMV(val_h, input_window, horizon_points)

        if len(ds_train) == 0 or len(ds_val) == 0:
            continue

        loaders_train.append(
            DataLoader(ds_train, batch_size=batch_size, shuffle=True)
        )
        loaders_val.append(
            DataLoader(ds_val, batch_size=batch_size, shuffle=False)
        )

    if not loaders_train:
        return float("inf")

    model = LSTMModel(
        input_size=INPUT_SIZE,
        hidden_size=hidden_size,
        num_layers=num_layers,
        horizon=horizon_points,
        dropout=dropout,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-5
    )
    criterion = nn.MSELoss()
    best_val = float("inf")
    patience_counter = 0

    for epoch in range(30):
        model.train()
        for loader in loaders_train:
            for x_b, y_b in loader:
                x_b, y_b = x_b.to(device), y_b.to(device)
                optimizer.zero_grad()
                criterion(model(x_b), y_b).backward()
                optimizer.step()

        model.eval()
        val_loss = 0
        n_val_batches = 0
        with torch.no_grad():
            for loader in loaders_val:
                for x_b, y_b in loader:
                    x_b, y_b = x_b.to(device), y_b.to(device)
                    val_loss += criterion(model(x_b), y_b).item()
                    n_val_batches += 1

        val_loss /= max(n_val_batches, 1)
        scheduler.step(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= 3:
            break

        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return best_val


print("Подбор гиперпараметров (все дома, горизонт 24h)")
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=3),
)
study.optimize(
    lambda trial: objective(trial, horizons["24h"]),
    n_trials=10,
    show_progress_bar=True,
)

best_params = study.best_params
print("\nЛучшие параметры:")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print(f"Лучший val_loss: {study.best_value:.6f}")

Подбор гиперпараметров (все дома, горизонт 24h)


  0%|          | 0/10 [00:00<?, ?it/s]


Лучшие параметры:
  hidden_size: 64
  num_layers: 3
  dropout: 0.23749489816302
  lr: 0.0028154964480084916
  batch_size: 32
Лучший val_loss: 0.004976


In [12]:
def predict_lstm(model, val_data, test_data, input_window, horizon, scaler, house):
    model.eval()
    predictions, actuals = [], []
    context = np.concatenate([val_data, test_data])
    n_f = HOUSE_META[house]['n_flats']

    with torch.no_grad():
        for i in range(0, len(context) - input_window - horizon + 1, horizon):
            x = context[i: i + input_window]
            y = context[i + input_window: i + input_window + horizon, 0]
            if len(y) < horizon:
                break
            x_tensor = torch.FloatTensor(x).unsqueeze(0).to(device)
            y_pred = model(x_tensor).cpu().numpy().flatten()
            predictions.extend(y_pred)
            actuals.extend(y)

    predictions = scaler.inverse_transform(
        np.array(predictions).reshape(-1, 1)
    ).flatten() * n_f
    actuals = scaler.inverse_transform(
        np.array(actuals).reshape(-1, 1)
    ).flatten() * n_f
    return actuals, predictions


def predict_lstm_long(model, train_data, val_data, test_data,
                      input_window, horizon, scaler, house):
    model.eval()
    predictions, actuals = [], []
    context = np.concatenate([train_data[-input_window:], val_data, test_data])
    n_f = HOUSE_META[house]['n_flats']

    with torch.no_grad():
        for i in range(0, len(context) - input_window - horizon + 1, horizon):
            x = context[i: i + input_window]
            y = context[i + input_window: i + input_window + horizon, 0]
            if len(y) < horizon:
                break
            x_tensor = torch.FloatTensor(x).unsqueeze(0).to(device)
            y_pred = model(x_tensor).cpu().numpy().flatten()
            predictions.extend(y_pred)
            actuals.extend(y)

    predictions = scaler.inverse_transform(
        np.array(predictions).reshape(-1, 1)
    ).flatten() * n_f
    actuals = scaler.inverse_transform(
        np.array(actuals).reshape(-1, 1)
    ).flatten() * n_f
    return actuals, predictions

In [13]:
lstm_results = {}
best_models = {}

for horizon_name, horizon_points in tqdm(horizons.items(), desc="LSTM горизонты"):

    house_loaders_train = []
    house_loaders_val = []
    for house, frame in all_house_data:
        train_h = frame[:train_end]
        val_h = frame[train_end:val_end]

        if len(train_h) <= input_window + horizon_points:
            continue
        if len(val_h) <= input_window + horizon_points:
            continue

        ds_train = TimeSeriesDatasetMV(train_h, input_window, horizon_points)
        ds_val = TimeSeriesDatasetMV(val_h, input_window, horizon_points)

        if len(ds_train) == 0 or len(ds_val) == 0:
            continue

        house_loaders_train.append(
            DataLoader(ds_train, batch_size=best_params["batch_size"], shuffle=True)
        )
        house_loaders_val.append(
            DataLoader(ds_val, batch_size=best_params["batch_size"], shuffle=False)
        )

    if not house_loaders_train:
        tqdm.write(f"Горизонт {horizon_name}: недостаточно данных, пропускаем")
        lstm_results[horizon_name] = {
            house: {"MAE": 0.0, "RMSE": 0.0, "MAPE": 0.0}
            for house, _ in all_house_data
        }
        continue

    model = LSTMModel(
        input_size=INPUT_SIZE,
        hidden_size=best_params["hidden_size"],
        num_layers=best_params["num_layers"],
        horizon=horizon_points,
        dropout=best_params["dropout"],
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=best_params["lr"])
    criterion = nn.MSELoss()
    best_val_loss = float("inf")
    patience_counter = 0
    best_weights = None

    for epoch in range(200):
        model.train()
        n_batches = 0
        for loader in house_loaders_train:
            for x_b, y_b in loader:
                x_b, y_b = x_b.to(device), y_b.to(device)
                optimizer.zero_grad()
                loss = criterion(model(x_b), y_b)
                loss.backward()
                optimizer.step()
                n_batches += 1

        model.eval()
        val_loss = 0
        n_val_batches = 0
        with torch.no_grad():
            for loader in house_loaders_val:
                for x_b, y_b in loader:
                    x_b, y_b = x_b.to(device), y_b.to(device)
                    val_loss += criterion(model(x_b), y_b).item()
                    n_val_batches += 1

        val_loss /= max(n_val_batches, 1)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= 10:
            break

    model.load_state_dict(best_weights)
    torch.save(model.state_dict(), f"lstm_{horizon_name}.pth")
    best_models[horizon_name] = model

    horizon_results = {}
    for house, frame in all_house_data:
        train_h = frame[:train_end]
        val_h = frame[train_end:val_end]
        test_h = frame[val_end:]
        scaler = scalers[house]

        if horizon_name == "1m":
            actuals, predictions = predict_lstm_long(
                model, train_h, val_h, test_h,
                input_window, horizon_points, scaler, house,
            )
        else:
            actuals, predictions = predict_lstm(
                model, val_h, test_h,
                input_window, horizon_points, scaler, house,
            )

        metrics = evaluate(actuals, predictions)
        horizon_results[house] = metrics

        n_pred = len(actuals)
        if n_pred == 0:
            continue

        ts_start = train_end + input_window
        ts_pred = df["timestamp"].iloc[ts_start: ts_start + n_pred].values

        if len(ts_pred) < n_pred:
            n_pred = len(ts_pred)
            actuals = actuals[:n_pred]
            predictions = predictions[:n_pred]

        if n_pred == 0:
            continue

        save_predictions(
            ts=ts_pred,
            house_ids=np.full(n_pred, house),
            y_true=actuals,
            y_pred=predictions,
            horizon_name=horizon_name,
            model_name="LSTM",
            filename="predictions_lstm.csv",
        )

        tqdm.write(f"{house} {horizon_name}: MAPE={metrics['MAPE']:.3f}%")

    lstm_results[horizon_name] = horizon_results

# Сводная таблица
rows = []
for horizon_name in horizons.keys():
    for house, _ in all_house_data:
        metrics = lstm_results[horizon_name][house]
        rows.append({
            "модель": "LSTM",
            "дом": house,
            "горизонт": horizon_name,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "MAPE": metrics["MAPE"],
        })

df_lstm = pd.DataFrame(rows)

for metric in ["MAE", "RMSE", "MAPE"]:
    pivot = df_lstm.pivot(index="горизонт", columns="дом", values=metric)
    pivot = pivot.reindex(list(horizons.keys()))
    pivot["Среднее"] = pivot.mean(axis=1).round(2)
    print(f"\nLSTM - {metric}:")
    print(pivot.round(3).to_string())

df_lstm.to_csv("results_lstm.csv", index=False)

LSTM горизонты:   0%|          | 0/6 [04:14<?, ?it/s]

house_1 4h: MAPE=11.840%


LSTM горизонты:   0%|          | 0/6 [04:15<?, ?it/s]

house_2 4h: MAPE=14.660%


LSTM горизонты:   0%|          | 0/6 [04:15<?, ?it/s]

...

house_6 1m: MAPE=11.221%


LSTM горизонты:  83%|████████▎ | 5/6 [23:50<03:49, 229.78s/it]

house_7 1m: MAPE=10.603%


LSTM горизонты: 100%|██████████| 6/6 [23:51<00:00, 238.61s/it]

house_8 1m: MAPE=9.186%

LSTM - MAE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8  Среднее
горизонт                                                                                 
4h          5.472    2.951    2.220    2.994    2.126    6.448    6.388    1.802     3.80
8h          6.341    3.146    2.360    3.493    2.402    7.749    7.334    2.058     4.36
24h         7.126    3.330    2.308    3.447    2.303    6.142    7.112    2.222     4.25
7d          8.553    4.037    2.876    4.074    2.565    8.649    8.978    2.559     5.29
14d         8.596    3.868    2.778    4.190    2.566    8.567    9.130    2.786     5.31
1m          7.582    3.503    2.324    4.362    2.285    8.543   10.059    2.883     5.19

LSTM - RMSE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8  Среднее
горизонт                                                                                 
4h          6.655    3.534    2.702    3.898    2

In [16]:
house_plot = "house_1"
frame_plot = dict(all_house_data)[house_plot]
train_plot = frame_plot[:train_end]
val_plot   = frame_plot[train_end:val_end]
test_plot  = frame_plot[val_end:]
scaler_plot = scalers[house_plot]

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f"Горизонт {h}" for h in horizons.keys()],
    vertical_spacing=0.1,
    horizontal_spacing=0.08,
)

for i, (horizon_name, horizon_points) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    if horizon_name not in best_models:
        continue

    model_h = best_models[horizon_name]

    if horizon_name == "1m":
        actuals_h, predictions_h = predict_lstm_long(
            model_h, train_plot, val_plot, test_plot,
            input_window, horizon_points, scaler_plot, house_plot,
        )
        ts_h = df["timestamp"].iloc[
            val_end: val_end + len(actuals_h)
        ].values
    else:
        actuals_h, predictions_h = predict_lstm(
            model_h, val_plot, test_plot,
            input_window, horizon_points, scaler_plot, house_plot,
        )
        ts_h = df["timestamp"].iloc[
            train_end + input_window: train_end + input_window + len(actuals_h)
        ].values

    n_show = min(horizon_points, len(actuals_h), len(ts_h))

    fig.add_trace(go.Scatter(
        x=ts_h[:n_show], y=actuals_h[:n_show],
        mode="lines", name="факт",
        line=dict(color="#1f77b4", width=1.2, shape="hv"),
        showlegend=(i == 0),
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=ts_h[:n_show], y=predictions_h[:n_show],
        mode="lines", name="прогноз LSTM",
        line=dict(color="#d62728", width=1.2, shape="hv"),
        showlegend=(i == 0),
    ), row=row, col=col)

    fig.update_xaxes(title_text="Дата", row=row, col=col, title_font=dict(size=9))
    fig.update_yaxes(title_text="кВт",  row=row, col=col, title_font=dict(size=9))

fig.update_layout(
    title="LSTM: прогнозные и фактические значения по всем горизонтам (дом 1)",
    width=700, height=900, font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9)),
)
fig.show()
fig.write_image("33_lstm_forecast_all_horizons.png", scale=2)